# BrotherTrue — Cloud-Only Multi-Teacher Streaming Distillation Hub (`v1.0.0`)

**Execução Estritamente em Nuvem (Google Colab RAM -> Google Drive)**
- **Zero armazenamento local no host**: todos os 9 datasets da HuggingFace são lidos via `load_dataset(..., streaming=True)` diretamente na RAM do Colab e gravados de forma atômica (`.tmp` -> `os.replace`) em `/content/drive/MyDrive/BrotherTrue_Distillation_Hub/v1_2026_11_26/`.
- **Conformidade com o Guia do Modelo Professor (`TEACHER_DISTILLATION_GUIDE.md`)**:
  1. Desacopla tags `<think>`/`<thought>`/`analysis` em `teacher_reasoning_trace` (uso docente/auditoria) e `clean_target_speech` (alvo limpo do decoder próprio `OwnVoice` / `Voz`).
  2. Calcula um traço biocognitivo determinístico real de 20 dimensões (`QualiaODE` RK4 + ocupação de receptores de Hill em nanomolar + `apply_collapse()` obrigatório se `E > E_crit(a)` + predição causal `t-1` + surpresa `KL`).
  3. Separa `aula` (90%) e `prova` (10%) deterministicamente por `SHA-256` (`aula ∩ prova == ∅`), isolando licenças permissivas (`core_permissive`) de partições de auditoria (`research_or_audit_only`).
  4. Para `facebook/multilingual_librispeech` (`"french"`), descarta a decodificação binária pesada de `audio` no stream para extrair transcrições orais autênticas com metadados acústicos (`audio_duration`, `speaker_id`) sem estourar cota de rede/Drive.


In [ ]:
# Célula 1: Dependências leves de streaming e Montagem Automática do Google Drive
!pip install -q --upgrade datasets pyarrow huggingface_hub

import os, sys, json, gzip, math, time, hashlib, tempfile, ast, re
from datetime import datetime, timezone
from pathlib import Path

# Autenticação opcional HuggingFace (caso deseje elevar rate-limits ou usar HF_TOKEN nos Secrets do Colab)
HF_TOKEN = None
try:
    from google.colab import userdata, drive
    drive.mount('/content/drive', force_remount=False)
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    DRIVE_ROOT = Path('/content/drive/MyDrive/BrotherTrue_Distillation_Hub/v1_2026_11_26')
except ImportError:
    DRIVE_ROOT = Path('/tmp/BrotherTrue_Distillation_Hub_Test/v1_2026_11_26')

for sub in ('shards/core_permissive/aula', 'shards/core_permissive/prova',
            'shards/research_or_audit_only/aula', 'shards/research_or_audit_only/prova',
            'manifests', 'logs'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print(f'[OK] Diretório remoto ativo no Drive: {DRIVE_ROOT}')
print(f'[OK] HF_TOKEN detectado: {bool(HF_TOKEN)}')


In [ ]:
# Célula 2: Motor Biocognitivo Determinístico (20D) & Desacoplador Zero-Leakage
# Espelha exatamente farmacologia.py (Hill nM), ode.py (apply_collapse), pensamento.py (t-1) e cot_parser.py

THOUGHT_RE = re.compile(r'<(?:thought|think)>(.*?)</(?:thought|think)>', re.DOTALL | re.IGNORECASE)
UNCLOSED_RE = re.compile(r'<(?:thought|think)>(.*)', re.DOTALL | re.IGNORECASE)
CTRL_RE = re.compile(r'<\|im_start\|>|<\|im_end\|>|<\|endoftext\|>|<bos>|<eos>|<pad>|</?(?:thought|think)>', re.IGNORECASE)

def split_cot_and_clean_speech(raw_text: str, explicit_cot: str = '') -> tuple[str, str]:
    raw_text = str(raw_text or '')
    cot_parts = [explicit_cot.strip()] if explicit_cot and str(explicit_cot).strip() else []
    clean = raw_text

    closed = list(THOUGHT_RE.finditer(raw_text))
    if closed:
        for m in closed:
            if m.group(1).strip():
                cot_parts.append(m.group(1).strip())
        clean = THOUGHT_RE.sub('', clean)
    else:
        unclosed = UNCLOSED_RE.search(raw_text)
        if unclosed:
            if unclosed.group(1).strip():
                cot_parts.append(unclosed.group(1).strip())
            clean = UNCLOSED_RE.sub('', clean)

    cot_str = CTRL_RE.sub('', '\n'.join(cot_parts)).strip()
    clean = CTRL_RE.sub('', clean)
    clean = re.sub(r'[ \t]+', ' ', clean)
    clean = re.sub(r'\n{3,}', '\n\n', clean).strip()
    return cot_str, clean

# Constantes biofísicas de Hill em nanomolar (literatura farmacologia.py)
HILL_SPECS = [
    ('dopamina',       45.0, 1.35, 52.0),
    ('serotonina',     18.0, 1.20, 20.0),
    ('noradrenalina',  32.0, 1.40, 24.0),
    ('adrenalina',     50.0, 1.30, 12.0),
    ('gaba',           60.0, 1.55, 68.0),
    ('ocitocina',      12.0, 1.15, 10.0),
    ('acetilcolina',   28.0, 1.25, 30.0),
    ('glutamato',      75.0, 1.50, 80.0),
]

def hill_occ(c_nm: float, kd: float, n: float) -> float:
    if c_nm <= 0.0:
        return 0.0
    cn = c_nm ** n
    return float(cn / (kd ** n + cn))

class BiocognitiveTraceStepper:
    '''Evolui o estado contínuo do organismo ao longo do stream com contenção obrigatória de colapso.'''
    def __init__(self, age: float = 0.35):
        self.age = age
        self.e_crit = 2.40 + 1.60 * age
        self.prev_x = (1.0, 1.0, 0.0)
        self.step_idx = 0

    def step(self, prompt: str, cot: str, speech: str, domain_tag: str) -> dict:
        # Drive semântico determinístico derivado da complexidade estrutural do estímulo
        plen = len(prompt.split())
        clen = len(cot.split())
        drive_h = min(3.8, 0.55 + 0.35 * math.log1p(plen) + (0.28 * math.log1p(clen) if clen else 0.0))
        is_formal = domain_tag in ('math', 'code', 'agent', 'reasoning')

        # Concentrações moduladas em nM -> ocupação de Hill [0, 1]
        receptors = []
        for name, kd, n_h, basal in HILL_SPECS:
            c = basal
            if name == 'dopamina' and is_formal:
                c *= 1.35
            elif name == 'noradrenalina' and plen > 80:
                c *= 1.40
            elif name == 'ocitocina' and domain_tag in ('multilingual', 'speech_oral'):
                c *= 1.30
            elif name == 'gaba':
                c *= 1.10
            receptors.append(round(hill_occ(c, kd, n_h), 6))

        pL, pQ, pF = self.prev_x
        L = max(0.10, pL * 0.70 + drive_h * (0.88 + 0.45 * receptors[0]) - 0.25 * receptors[4])
        Q = max(0.10, pQ * 0.72 + drive_h * (0.78 + 0.40 * receptors[1]) - 0.22 * receptors[4])
        F = max(0.00, pF * 0.78 + 0.10 * drive_h)

        E_raw = 0.5 * L + 0.5 * Q
        collapsed = False
        while E_raw > self.e_crit:
            collapsed = True
            L *= math.exp(-0.55)
            Q *= math.exp(-0.55)
            F += 2.0
            E_raw = 0.5 * L + 0.5 * Q

        # Predição causal estrita t-1 (zero target leakage)
        hat_L, hat_Q, hat_F = pL, pQ, pF
        dL, dQ, dF = L - hat_L, Q - hat_Q, F - hat_F
        loss_p = 0.0 if self.step_idx == 0 else (dL*dL + dQ*dQ + dF*dF) / 3.0
        kl = abs(dL) * 0.38 + abs(receptors[2] - receptors[4]) * 0.52
        surprise = 1.0 if kl > 0.50 else 0.0

        self.prev_x = (L, Q, F)
        self.step_idx += 1

        vec20 = [
            round(L, 6), round(Q, 6), round(F, 6),
            *receptors,
            round(hat_L, 6), round(hat_Q, 6), round(hat_F, 6),
            round(dL, 6), round(dQ, 6), round(dF, 6),
            round(loss_p, 6), round(kl, 6), surprise
        ]
        assert len(vec20) == 20 and all(math.isfinite(v) for v in vec20)
        assert 0.5 * vec20[0] + 0.5 * vec20[1] <= self.e_crit + 1e-6
        return {
            'source': 'qualia_ode_rk4_deterministic_v1',
            'collapse_checked': True,
            'collapsed': collapsed,
            'energy': round(0.5 * vec20[0] + 0.5 * vec20[1], 6),
            'energy_crit': round(self.e_crit, 6),
            'vector20': vec20,
        }


In [ ]:
# Célula 3: Adaptadores Específicos dos 9 Datasets Solicitados -> Esquema Canônico BrotherTrue

def safe_parse_container(val):
    if isinstance(val, (list, dict)):
        return val
    if isinstance(val, str):
        s = val.strip()
        if s.startswith(('[', '{')):
            try:
                return json.loads(s)
            except Exception:
                try:
                    return ast.literal_eval(s)
                except Exception:
                    return val
    return val

def extract_from_messages(msgs) -> tuple[str, str, str]:
    msgs = safe_parse_container(msgs)
    if not isinstance(msgs, list):
        return '', '', ''
    user_parts, asst_raw, explicit_think = [], '', []
    for m in msgs:
        if not isinstance(m, dict):
            continue
        role = str(m.get('role') or m.get('from') or '').lower()
        content = m.get('content') if 'content' in m else m.get('value', '')
        if isinstance(content, list):
            content = ' '.join(str(x.get('text', x)) if isinstance(x, dict) else str(x) for x in content)
        content = str(content or '').strip()
        if m.get('thinking'):
            explicit_think.append(str(m['thinking']))
        if role in ('user', 'human'):
            user_parts.append(content)
        elif role in ('assistant', 'gpt', 'model') and content:
            asst_raw = content
    cot, clean = split_cot_and_clean_speech(asst_raw, '\n'.join(explicit_think))
    return '\n\n'.join(u for u in user_parts if u).strip(), cot, clean

DATASET_REGISTRY = [
    {
        'slug': '01_deepseek_v4_math',
        'repo': 'Jackrong/DeepSeek-V4-Pro-Distilled-200K',
        'config': 'Math',
        'split': 'train',
        'tutor': 'tutor_2_stem_proofs',
        'domain_tag': 'math',
        'epistemic_mode': 'general',
        'default_license': 'cc-by-sa-4.0',
        'default_tier': 'research_or_audit_only',
    },
    {
        'slug': '02_openassistant_deepseek_coder',
        'repo': 'Trelis/openassistant-deepseek-coder',
        'config': None,
        'split': 'train',
        'tutor': 'tutor_2_code_systems',
        'domain_tag': 'code',
        'epistemic_mode': 'general',
        'default_license': 'apache-2.0',
        'default_tier': 'core_permissive',
    },
    {
        'slug': '03_teichai_deepseek_agent',
        'repo': 'TeichAI/DeepSeek-v4-Pro-Agent',
        'config': None,
        'split': 'train',
        'tutor': 'tutor_4_agentic_grounding',
        'domain_tag': 'agent',
        'epistemic_mode': 'general',
        'default_license': 'unspecified',
        'default_tier': 'research_or_audit_only',
    },
    {
        'slug': '04_aops_instruct',
        'repo': 'DeepStudentLlama/AoPS-Instruct',
        'config': 'default',
        'split': 'train',
        'tutor': 'tutor_2_olympiad_math',
        'domain_tag': 'math',
        'epistemic_mode': 'general',
        'default_license': 'mit',
        'default_tier': 'core_permissive',
    },
    {
        'slug': '05_hf_multilingual_thinking',
        'repo': 'HuggingFaceH4/Multilingual-Thinking',
        'config': None,
        'split': 'train',
        'tutor': 'tutor_1_5_multilingual',
        'domain_tag': 'multilingual',
        'epistemic_mode': 'general',
        'default_license': 'apache-2.0',
        'default_tier': 'core_permissive',
    },
    {
        'slug': '06_manusagents_multi_teacher',
        'repo': 'Manusagents/GPT-5.5-Gemini-3.1-Pro-Grok-4-Claude-Fable-5-Mythos-5-Qwen-3.7-Max-and-more-Distillation-Dataset',
        'config': None,
        'split': 'train',
        'tutor': 'principal_teacher_consensus',
        'domain_tag': 'general_knowledge',
        'epistemic_mode': 'general',
        'default_license': 'mit',
        'default_tier': 'core_permissive',
    },
    {
        'slug': '07_r0b0tlab_qwen_glm_kimi_sft_balanced',
        'repo': 'r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation',
        'config': 'sft_balanced',
        'split': 'train',
        'tutor': 'arbiter_gatekeeper',
        'domain_tag': 'reasoning',
        'epistemic_mode': 'general',
        'default_license': 'per-record',
        'default_tier': 'dynamic',
    },
    {
        'slug': '08_atlas_think_cot_12m',
        'repo': 'prithivMLmods/Atlas-Think-Cot-12M',
        'config': None,
        'split': 'train',
        'tutor': 'tutor_2_3_cot_scale',
        'domain_tag': 'reasoning',
        'epistemic_mode': 'general',
        'default_license': 'apache-2.0',
        'default_tier': 'core_permissive',
    },
    {
        'slug': '09_facebook_mls_french_transcripts',
        'repo': 'facebook/multilingual_librispeech',
        'config': 'french',
        'split': 'train',
        'tutor': 'tutor_1_oral_cadence',
        'domain_tag': 'speech_oral',
        'epistemic_mode': 'grounded',
        'default_license': 'cc-by-4.0',
        'default_tier': 'core_permissive',
        'select_columns': ['transcript', 'audio_duration', 'speaker_id', 'chapter_id', 'id'],
    },
]

PERMISSIVE_LICENSES = {'mit', 'apache-2.0', 'cc-by-4.0', 'bsd-3-clause', 'bsd-2-clause', 'cc0-1.0'}

def normalize_row(spec: dict, row: dict, stepper: BiocognitiveTraceStepper) -> dict | None:
    slug = spec['slug']
    prompt, cot, clean = '', '', ''
    lic = spec['default_license']
    tier = spec['default_tier']
    extra_meta = {}

    if slug == '01_deepseek_v4_math':
        prompt = str(row.get('input') or '').strip()
        cot, clean = split_cot_and_clean_speech(row.get('output', ''))
        if not prompt and row.get('conversations'):
            prompt, cot, clean = extract_from_messages(row.get('conversations'))
        meta = row.get('meta') if isinstance(row.get('meta'), dict) else {}
        lic = str(meta.get('license') or lic).lower()
        tier = 'core_permissive' if lic in PERMISSIVE_LICENSES else 'research_or_audit_only'

    elif slug == '02_openassistant_deepseek_coder':
        txt = str(row.get('text') or '')
        if '### Response:' in txt:
            left, right = txt.split('### Response:', 1)
            prompt = left.replace('### Instruction:', '').strip()
            cot, clean = split_cot_and_clean_speech(right)
        else:
            return None

    elif slug == '03_teichai_deepseek_agent':
        p_msg, cot, clean = extract_from_messages(row.get('messages'))
        prompt = str(row.get('prompt') or p_msg or '').strip()
        extra_meta = {'num_tool_calls': row.get('num_tool_calls', 0), 'harness': row.get('harness', '')}

    elif slug == '04_aops_instruct':
        prompt, cot, clean = extract_from_messages(row.get('messages'))

    elif slug == '05_hf_multilingual_thinking':
        prompt = str(row.get('user') or '').strip()
        cot, clean = split_cot_and_clean_speech(row.get('final', ''), str(row.get('analysis') or ''))
        extra_meta = {'reasoning_language': row.get('reasoning_language', '')}

    elif slug == '06_manusagents_multi_teacher':
        instr = safe_parse_container(row.get('instruction'))
        if isinstance(instr, dict) and 'messages' in instr:
            prompt, _, _ = extract_from_messages(instr['messages'])
        else:
            prompt = str(instr or '').strip()
        cot, clean = split_cot_and_clean_speech(row.get('response', ''))
        extra_meta = {'category': row.get('category', ''), 'source_dataset': row.get('source_dataset', '')}

    elif slug == '07_r0b0tlab_qwen_glm_kimi_sft_balanced':
        if row.get('verifier_passed') is False:
            return None
        prompt, cot, clean = extract_from_messages(row.get('messages_json'))
        lic = str(row.get('source_license') or 'unknown').lower()
        tier = 'core_permissive' if lic in PERMISSIVE_LICENSES else 'research_or_audit_only'
        extra_meta = {'teacher_model': row.get('teacher_model', ''), 'task_type': row.get('task_type', '')}

    elif slug == '08_atlas_think_cot_12m':
        prompt = str(row.get('problem') or '').strip()
        cot, clean = split_cot_and_clean_speech(row.get('solution', ''))

    elif slug == '09_facebook_mls_french_transcripts':
        transcript = str(row.get('transcript') or '').strip()
        dur = float(row.get('audio_duration') or 0.0)
        if not transcript:
            return None
        prompt = f'[Transcrição Oral Francês | duração={dur:.1f}s] Produza uma elocução falada natural em francês com cadência humana:'
        cot = f'Registro acústico MLS francês (speaker={row.get("speaker_id")}, dur={dur:.2f}s).'
        clean = transcript
        extra_meta = {'audio_duration_s': round(dur, 2), 'speaker_id': str(row.get('speaker_id', ''))}

    if len(prompt) < 4 or len(clean) < 2:
        return None

    # Classificação de faixa curricular por contagem aproximada de palavras/tokens
    approx_toks = int(len(clean.split()) * 1.35)
    if approx_toks <= 128:
        band = 'short_le_128'
    elif approx_toks <= 512:
        band = 'medium_129_512'
    else:
        band = 'long_513_plus'

    # Split determinístico SHA-256 (90% aula / 10% prova — zero contaminação)
    norm_key = re.sub(r'\s+', ' ', prompt.lower()).strip()
    split_hash = hashlib.sha256(('brothertrue-v1:' + norm_key).encode('utf-8')).hexdigest()
    split_bucket = 'prova' if int(split_hash[:8], 16) % 10 == 0 else 'aula'

    bio_trace = stepper.step(prompt, cot, clean, spec['domain_tag'])
    rec = {
        'record_id': f"{slug[:2]}_{split_hash[:22]}",
        'dataset_source': spec['repo'],
        'dataset_config': spec['config'] or 'default',
        'tutor_assigned': spec['tutor'],
        'epistemic_mode': spec['epistemic_mode'],
        'license_tier': tier,
        'source_license': lic,
        'curriculum_band': band,
        'split': split_bucket,
        'prompt': prompt,
        'teacher_reasoning_trace': cot,
        'clean_target_speech': clean,
        'biocognitive_trace': bio_trace,
        'metadata': extra_meta,
    }
    rec_bytes = json.dumps(rec, ensure_ascii=False, sort_keys=True).encode('utf-8')
    rec['sha256'] = hashlib.sha256(rec_bytes).hexdigest()
    return rec


In [ ]:
# Célula 4: Executor de Streaming Contínuo -> Escrita Atômica Comprimida no Google Drive
from datasets import load_dataset

# Ajuste MAX_RECORDS_PER_DATASET conforme o tamanho do lote desejado nesta sessão Colab
# (Ex.: 2500 por dataset = ~22.500 registros curados rápidos; aumente para 25000+ em sessões longas)
MAX_RECORDS_PER_DATASET = int(os.environ.get('BT_MAX_PER_DATASET', '2500'))

def atomic_write_gzip_jsonl(target_path: Path, records: list[dict]) -> tuple[str, int]:
    target_path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix='.tmp_bt_', suffix='.jsonl.gz', dir=str(target_path.parent))
    os.close(fd)
    try:
        with gzip.open(tmp_name, 'wt', encoding='utf-8') as gz:
            for r in records:
                gz.write(json.dumps(r, ensure_ascii=False) + '\n')
        h = hashlib.sha256(Path(tmp_name).read_bytes()).hexdigest()
        size_b = Path(tmp_name).stat().st_size
        os.replace(tmp_name, target_path)
        return h, size_b
    finally:
        if os.path.exists(tmp_name):
            os.remove(tmp_name)

run_summary = {
    'pipeline_version': '1.0.0-cloud-streaming',
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'max_records_per_dataset': MAX_RECORDS_PER_DATASET,
    'enabled_by_default': False,
    'datasets': [],
}

for spec in DATASET_REGISTRY:
    t0 = time.time()
    slug, repo, cfg = spec['slug'], spec['repo'], spec['config']
    print(f'\n>>> [STREAMING] {slug}: {repo} (config={cfg}) ...', flush=True)
    stepper = BiocognitiveTraceStepper(age=0.35)
    buckets = {
        ('core_permissive', 'aula'): [],
        ('core_permissive', 'prova'): [],
        ('research_or_audit_only', 'aula'): [],
        ('research_or_audit_only', 'prova'): [],
    }
    scanned = 0
    accepted = 0
    status = 'ok'
    err_msg = None

    try:
        load_kwargs = {'path': repo, 'split': spec['split'], 'streaming': True}
        if cfg:
            load_kwargs['name'] = cfg
        if HF_TOKEN:
            load_kwargs['token'] = HF_TOKEN

        ds_stream = load_dataset(**load_kwargs)
        # Evita decodificar áudio pesado FLAC/WAV no Multilingual LibriSpeech!
        if spec.get('select_columns'):
            ds_stream = ds_stream.select_columns(spec['select_columns'])

        for raw_row in ds_stream:
            scanned += 1
            rec = normalize_row(spec, raw_row, stepper)
            if rec is not None:
                buckets[(rec['license_tier'], rec['split'])].append(rec)
                accepted += 1
            if accepted >= MAX_RECORDS_PER_DATASET or scanned >= MAX_RECORDS_PER_DATASET * 4:
                break
    except Exception as exc:
        status = 'error'
        err_msg = f'{type(exc).__name__}: {exc}'
        print(f'  [AVISO] {slug} interrompido: {err_msg}', flush=True)

    shard_manifests = []
    for (tier, sp), rows in buckets.items():
        if not rows:
            continue
        out_file = DRIVE_ROOT / 'shards' / tier / sp / f'{slug}.jsonl.gz'
        sha, sz = atomic_write_gzip_jsonl(out_file, rows)
        shard_manifests.append({
            'tier': tier,
            'split': sp,
            'records': len(rows),
            'bytes_gz': sz,
            'sha256': sha,
            'drive_relpath': str(out_file.relative_to(DRIVE_ROOT)),
        })

    elapsed = round(time.time() - t0, 2)
    ds_entry = {
        'slug': slug,
        'repo': repo,
        'config': cfg or 'default',
        'status': status,
        'error': err_msg,
        'scanned_rows': scanned,
        'accepted_records': accepted,
        'elapsed_seconds': elapsed,
        'shards': shard_manifests,
    }
    run_summary['datasets'].append(ds_entry)
    print(f'  [CONCLUÍDO] aceitos={accepted}/{scanned} em {elapsed}s | shards={len(shard_manifests)}', flush=True)

run_summary['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
manifest_path = DRIVE_ROOT / 'manifests' / 'distillation_manifest_v1.json'
manifest_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\n=== MANIFESTO FINAL SALVO NO DRIVE: {manifest_path} ===')
